# ControlNet — Sketch to Image

Independent Colab/T4 experiment. Requires a GPU runtime and internet access for model weights. Generates `/content/generated-chair.png`. This notebook deliberately excludes TripoSR code.

Models: `lllyasviel/control_v11p_sd15_scribble` and `stable-diffusion-v1-5/stable-diffusion-v1-5` (pretrained external resources). Record their licenses and attribution in the final report.

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.10"], check=True)

In [ ]:
import subprocess
from pathlib import Path

env = Path("/content/controlnet-env/bin/python")
if not env.exists():
    subprocess.run(["uv", "venv", "/content/controlnet-env", "--python", "3.10"], check=True)
print("ControlNet Python:", env.exists())

In [ ]:
import subprocess
subprocess.run([
    "uv", "pip", "install", "--python", "/content/controlnet-env/bin/python",
    "torch==2.1.2", "torchvision==0.16.2",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

In [ ]:
import subprocess
subprocess.run([
    "uv", "pip", "install", "--python", "/content/controlnet-env/bin/python",
    "numpy==1.26.4", "diffusers==0.27.2", "transformers==4.38.2",
    "accelerate==0.29.3", "huggingface-hub==0.25.2", "safetensors", "Pillow==10.2.0"
], check=True)

In [ ]:
import subprocess
subprocess.run([
    "/content/controlnet-env/bin/python", "-c",
    "import torch; assert torch.cuda.is_available(), 'GPU unavailable'; print(torch.__version__, torch.cuda.get_device_name(0))"
], check=True)

## Upload and preprocess sketch

Upload a dark-on-light sketch. The control image is white strokes on black, centered in 512×512 without stretching.

In [ ]:
from google.colab import files

uploaded = files.upload()
sketch_filename = next(iter(uploaded))

print("Uploaded:", sketch_filename)

In [ ]:
from PIL import Image, ImageOps
import numpy as np

sketch = Image.open(sketch_filename).convert("L")
sketch = ImageOps.autocontrast(sketch)

arr = np.asarray(sketch)

# Identify dark sketch strokes.
mask = arr < 200

if not mask.any():
    raise ValueError("No dark sketch lines found!")

ys, xs = np.where(mask)

left, right = xs.min(), xs.max() + 1
top, bottom = ys.min(), ys.max() + 1

cropped = sketch.crop((left, top, right, bottom))

# Resize without distorting aspect ratio.
cropped.thumbnail((440, 440), Image.Resampling.LANCZOS)

canvas = Image.new("L", (512, 512), 255)

x = (512 - cropped.width) // 2
y = (512 - cropped.height) // 2

canvas.paste(cropped, (x, y))

# White strokes on black background.
control = Image.fromarray(
    np.where(np.asarray(canvas) < 200, 255, 0).astype("uint8")
).convert("RGB")

control.save("/content/control.png")

display(canvas)
display(control)

## Generate image

Edit the `prompt` inside the next cell for the uploaded object. `enable_model_cpu_offload()` requires a GPU-enabled runtime.

In [ ]:
%%writefile /content/generate_sketch_image.py

import time
import torch
from PIL import Image
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler
)

start = time.time()

control_image = Image.open(
    "/content/control.png"
).convert("RGB")

print("Loading ControlNet...")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_scribble",
    torch_dtype=torch.float16
)

print("Loading Stable Diffusion...")

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config
)

# Reduce T4 GPU memory pressure.
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

prompt = (
    "a single wooden chair, realistic product photography, "
    "centered isolated object, full object visible, "
    "clean neutral gray background, realistic material, "
    "accurate proportions, detailed three-dimensional appearance"
)

negative_prompt = (
    "multiple chairs, people, clutter, text, watermark, "
    "cropped object, floating parts, distorted geometry"
)

print("Generating image...")

generator = torch.Generator(
    device="cpu"
).manual_seed(42)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=control_image,
    num_inference_steps=20,
    guidance_scale=7.0,
    controlnet_conditioning_scale=1.1,
    generator=generator
).images[0]

image.save("/content/generated-chair.png")

print("Saved: /content/generated-chair.png")
print("Total elapsed seconds:", round(time.time()-start, 2))

In [ ]:
import subprocess

subprocess.run(
    [
        "/content/controlnet-env/bin/python",
        "/content/generate_sketch_image.py"
    ],
    check=True
)

In [ ]:
from pathlib import Path
from PIL import Image

image_path = Path("/content/generated-chair.png")

assert image_path.is_file(), "ControlNet image was not generated"

display(Image.open(image_path))
print("SUCCESS: Generated image exists")

In [ ]:
from google.colab import files
files.download("/content/generated-chair.png")